In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import pprint
# Base URL for Metacritic's 2024 games page
base_url = "https://www.metacritic.com/browse/game/all/all/2024/new/?page="
game_base_url = "https://www.metacritic.com"

# Headers to mimic a browser request (Metacritic may block requests without headers)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

# Initialize a list to hold game data
games_data = []
games_link_list = []

# Loop through pages (adjust range for the number of pages to scrape)
for page in range(1, 1000):  # Assuming there are about 1000 pages
    print(f"Scraping page {page}...")
    url = f"{base_url}{page}"

    # Send a GET request to the URL
    response = requests.get(url, headers=headers)

    # Check if the request was successful
    if response.status_code == 200:
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.content, 'html.parser')

        # Find all game entries on the page
        for link in  soup.find_all('a', class_='c-finderProductCard_container g-color-gray80 u-grid'):
            games_link_list.append(game_base_url + link.get('href'))
        
    else:
        print(f"Failed to scrape page {page}. HTTP Status Code: {response.status_code}")
        break




In [ ]:
# Loop through the list of game links
for link in games_link_list:
    # Extract game details
    response = requests.get(link, headers=headers)
    # Check if the request was successful
    if response.status_code == 200:
        print(f"Scraping game {link}...")
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.content, 'html.parser')

        # Extract game details
        title = soup.find('h1').text
        summary = soup.find('span', class_='c-productionDetailsGame_description g-text-xsmall').text
        platforms = []
        initial_release_date = None
        developers = []
        publisher = None
        genres = []
        metascore = None
        metascore_rate = None
        exact_metascore = None
        user_score = None
        user_score_rate = None
        exact_user_score = None
        age_rate = None
        age_name = None
        
        for platform in soup.find('ul', class_='g-outer-spacing-left-medium-fluid').find_all('li'):
            platforms.append(platform.text.replace(" ", "").replace("\n", ""))
        spans = soup.find_all('span', class_='g-text-bold u-block')        
        
        for span in spans:
            if span.text == "Initial Release Date:":
                initial_release_date = span.find_next_sibling("span").text
            if span.text == "Developer:": 
                for li in span.find_next_sibling("ul").find_all('li'):
                    developers.append(li.text)
            if span.text == "Publisher:": 
                publisher = span.find_next_sibling("span").text
            if span.text == "Genres:": 
                for li in span.find_next_sibling("ul").find_all('li'):
                    genres.append(li.text)

        if soup.find_all('div', class_="c-productScoreInfo_scoreContent u-flexbox u-flexbox-alignCenter u-flexbox-justifyFlexEnd g-width-100 u-flexbox-nowrap"):
            scores_html = soup.find_all('div', class_="c-productScoreInfo_scoreContent u-flexbox u-flexbox-alignCenter u-flexbox-justifyFlexEnd g-width-100 u-flexbox-nowrap")

            if len(scores_html) >= 1 and scores_html[0].find('span', class_="c-productScoreInfo_scoreSentiment g-text-bold g-color-gray90 g-text-xsmall"):
                metascore = scores_html[0].find('span', class_="c-productScoreInfo_scoreSentiment g-text-bold g-color-gray90 g-text-xsmall").text
            if len(scores_html) >= 1 and scores_html[0].find('a', class_="g-color-gray80 u-text-underline"):
                if scores_html[0].find('a', class_="g-color-gray80 u-text-underline").find('span'):
                    metascore_rate = scores_html[0].find('a', class_="g-color-gray80 u-text-underline").find('span').text
            if len(scores_html) >= 1 and scores_html[0].find('div', class_="c-productScoreInfo_scoreNumber u-float-right"):
                if scores_html[0].find('div', class_="c-productScoreInfo_scoreNumber u-float-right").find('div'):
                    if scores_html[0].find('div', class_="c-productScoreInfo_scoreNumber u-float-right").find('div').find('div'):
                        if scores_html[0].find('div', class_="c-productScoreInfo_scoreNumber u-float-right").find('div').find('div').find('span'):
                            exact_metascore = scores_html[0].find('div', class_="c-productScoreInfo_scoreNumber u-float-right").find('div').find('div').find('span').text

            if len(scores_html) >= 2 and scores_html[1].find('span', class_="c-productScoreInfo_scoreSentiment g-text-bold g-color-gray90 g-text-xsmall"):
                user_score = scores_html[1].find('span', class_="c-productScoreInfo_scoreSentiment g-text-bold g-color-gray90 g-text-xsmall").text
            if len(scores_html) >= 2 and scores_html[1].find('a', class_="g-color-gray80 u-text-underline"):
                if scores_html[1].find('a', class_="g-color-gray80 u-text-underline").find('span'):
                    user_score_rate = scores_html[1].find('a', class_="g-color-gray80 u-text-underline").find('span').text
            if len(scores_html) >= 2 and scores_html[1].find('div', class_="c-productScoreInfo_scoreNumber u-float-right"):
                if scores_html[1].find('div', class_="c-productScoreInfo_scoreNumber u-float-right").find('div'):
                    if scores_html[1].find('div', class_="c-productScoreInfo_scoreNumber u-float-right").find('div').find('div'):
                        if scores_html[1].find('div', class_="c-productScoreInfo_scoreNumber u-float-right").find('div').find('div').find('span'):
                            exact_user_score = scores_html[1].find('div', class_="c-productScoreInfo_scoreNumber u-float-right").find('div').find('div').find('span').text
                

        age_rating_div = soup.find('div', class_="c-productionDetailsGame_esrb_title u-inline-block g-outer-spacing-left-medium-fluid")
        
        if age_rating_div:
            if age_rating_div.find_all('span'):
                spans = age_rating_div.find_all('span')
                if len(spans) >= 1:
                    age_rate = spans[0].text
                if len(spans) >= 2:
                    age_name = spans[1].text

        games_data.append({
            "title": title, 
            "summary": summary, 
            "initial_release_date": initial_release_date, 
            "developers": developers, 
            "publisher": publisher, 
            "platforms": platforms, 
            "genres": genres, 
            "metascore": metascore,
            "exact_metascore": exact_metascore,
            "metascore_rate": metascore_rate,
            "user_score_rate": user_score_rate,
            "user_score": user_score,
            "exact_user_score": exact_user_score,
            "age_rate": age_rate,
            "age_name": age_name,
            })

df = pd.DataFrame(games_data)

df = df.applymap(
    lambda x: [y.strip() for y in x] if isinstance(x, list) and all(isinstance(y, str) for y in x)
    else x.strip() if isinstance(x, str)
    else x
)

df.to_csv('Games_2024_metacritic.csv', index=False)


